# 实验 1：3 种模型横向对比

**目的**：在统一数据集与评估协议下，横向对比 3 条无监督检测技术路线（计划书 4.3 对比实验）。
（初版实现 5 种路线后按测试集准确率筛选，保留前 3 名。）

**模型**：PatchCore（主方案）/ PaDiM / CutPaste

**指标**：图像级 AUROC、PixelAUROC、单张平均推理耗时（约束 <1s）

**输出**：`results/comparison.csv`、`comparison.md`、`results/heatmaps/` 热力图样例

**运行提示**：
- 本 Notebook 单独成册，可从 Jupyter 首页直接打开运行
- 权重缓存在 `checkpoints/`，重跑自动跳过训练；把第 ③ 节的 `RETRAIN = True` 可从头训练 3 个模型（约数小时）
- 若之前在同一内核跑过其他 Notebook，建议先 `Kernel → Restart & Run All` 保证环境变量先生效

## ① 环境准备

In [ ]:
import os
import sys
import inspect
from pathlib import Path

# 必须先于 numpy/torch 导入设置（Windows：防 OpenBLAS 内存报错、GBK 编码报错）
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")


def _find_project_root() -> Path:
    """定位项目根目录（包含 src/ 与 configs/ 的 defect_detection 目录）。"""
    # 方式 1：由 Notebook 文件所在位置向上找（VS Code / JupyterLab 均适用，不依赖启动目录）
    for var in ("__vsc_ipynb_file__", "__file__"):
        p = globals().get(var)
        if p and str(p).endswith(".ipynb"):
            for parent in Path(p).resolve().parents:
                if (parent / "src").exists() and (parent / "configs").exists():
                    return parent
    # 方式 2：从当前工作目录向上找（经典 Jupyter Notebook 场景）
    d = Path.cwd().resolve()
    for parent in (d, *d.parents):
        if (parent / "src").exists() and (parent / "configs").exists():
            return parent
    raise FileNotFoundError(
        "未找到项目根目录（defect_detection，含 src/ 与 configs/）。\n"
        "请确认 Notebook 位于项目的 experiments/ 目录下，或从项目根目录启动 Jupyter。"
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("项目根目录:", PROJECT_ROOT)
print("CUDA 可用:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## ② 实验配置与数据准备

In [ ]:
# ============ 训练开关 ============
RETRAIN = False   # True = 忽略已有权重、从头训练（耗时数小时）；False = 复用 checkpoints 仅评估

import json

import pandas as pd

from src.experiment_utils import get_or_train, load_config, release_memory, resolve
from src.data import get_test_loader, get_train_loader
from src.utils import set_seed

cfg = load_config()
set_seed(cfg["dataset"]["seed"])
data_root = resolve(cfg["paths"]["data_root"])
results_dir = resolve(cfg["paths"]["results_dir"])

print("数据根目录:", data_root)
print("图像尺寸:", cfg["dataset"]["image_size"], "| batch_size:", cfg["dataset"]["batch_size"])
print("本实验涉及模型的超参数（configs/experiment.yaml）:")
print(json.dumps(cfg["models"], indent=2, ensure_ascii=False))

# 测试集加载器（评估用，固定不变）
# Windows Jupyter 下多进程 DataLoader 不稳，统一 num_workers=0（不影响结果，仅加载略慢）
test_loader = get_test_loader(data_root, batch_size=cfg["dataset"]["batch_size"],
                              num_workers=0, image_size=cfg["dataset"]["image_size"])

## ③ 模型构建与训练（3 模型串行）

In [ ]:
from src.evaluation import evaluate_model, make_comparison_table
from src.models import build_model

MODELS = ["patchcore", "padim", "cutpaste"]

results = []
for name in MODELS:
    print(f"\n{'=' * 60}\n>>> 模型：{name}\n{'=' * 60}")
    # 每个模型单独创建训练加载器（DataLoader 为迭代器，不可复用）
    train_loader = get_train_loader(data_root, batch_size=cfg["dataset"]["batch_size"],
                                    num_workers=0, image_size=cfg["dataset"]["image_size"],
                                    use_augmentation=True)
    model = build_model(name, device="auto", image_size=cfg["dataset"]["image_size"],
                        **cfg["models"].get(name, {}))
    model = get_or_train(model, f"{name}.pth", train_loader, retrain=RETRAIN)  # ← 训练/加载权重
    res = evaluate_model(model, test_loader, heatmap_dir=results_dir / "heatmaps",
                         n_heatmap_samples=3)
    results.append(res)
    release_memory(model)   # 串行训练，及时释放显存

df = make_comparison_table(results, results_dir)
df

## ④ 训练源码速览（本实验模型的 train 实现）

以下用 `inspect` 直接打印项目 `src/` 中的训练代码，与上面实际执行的训练逻辑完全一致（单一来源，无需手工复制）：
- `get_or_train`：统一训练入口（权重缓存判断 / 训练调度 / 保存）
- 各模型 `train`：本实验的训练循环实现

In [ ]:
print(inspect.getsource(get_or_train))   # 统一训练入口：权重缓存 / 训练 / 保存

from src.models.cutpaste import CutPasteModel
from src.models.padim_anomalib import PadimAnomalibModel
from src.models.patchcore import PatchCoreModel

for cls in (PatchCoreModel, PadimAnomalibModel, CutPasteModel):
    print("\n" + "#" * 70)
    print(f"# {cls.__name__}.train  ——  {cls.__module__}")
    print("#" * 70)
    print(inspect.getsource(cls.train))

## 输出文件

`results/comparison.csv`（对比表）、`results/comparison.md`、`results/heatmaps/<模型>/`（三联图样例）。